In [2]:
import re

with open("scrape_log.txt", "r") as f:
    logs = f.read()

# Find all failed detail pages like "Failed to fetch detail page 2574"
details_pages = list(map(int, re.findall(r"Fetching details page (\d+)", logs)))
print(len(details_pages))


1


In [13]:
# Find all failed detail pages like "Failed to fetch detail page 2574"
skip_pages = list(map(int, re.findall(r"Skipping duplicate link: (\d+)", logs)))
print(len(skip_pages))

160


In [14]:
# Find all failed detail pages like "Failed to fetch detail page 2574"
fetched_pages = list(map(int, re.findall(r"Fetching page (\d+)...", logs)))
print(len(fetched_pages))

159


In [8]:
import pandas as pd
import re

# Load CSV that has broken rows
df = pd.read_csv("Flats.csv", header=None, dtype=str)

# Define a regex to detect a second URL in a row (e.g., two listings in one row)
def has_multiple_links(cell):
    if isinstance(cell, str):
        return len(re.findall(r'https?://', cell)) > 1
    return False

# Iterate and detect rows that contain multiple listings merged
rows_to_split = []
for idx, row in df.iterrows():
    combined_text = ' '.join(row.dropna().astype(str).tolist())
    if has_multiple_links(combined_text):
        rows_to_split.append(idx)

print(f"⚠️ Rows to split (probably merged listings): {rows_to_split}")

# Fix merged rows by splitting the row into two
fixed_rows = []
for idx, row in df.iterrows():
    combined = ','.join(row.dropna().astype(str).tolist())

    if idx in rows_to_split:
        # Try splitting based on known pattern: second occurrence of ₹ (INR symbol)
        parts = re.split(r'(₹\d[\d,\.]*\s*Lac)', combined)
        if len(parts) > 2:
            # Rebuild both parts
            first_half = ','.join(parts[:2]) + ',' + ','.join(parts[2:]).split(',')[0]  # crude but works
            second_half = ','.join(parts[2:])[len(','.join(parts[2:]).split(',')[0]) + 1:]
            fixed_rows.append(first_half.split(','))
            fixed_rows.append(second_half.split(','))
        else:
            # If failed to split, just append as is
            fixed_rows.append(row.tolist())
    else:
        fixed_rows.append(row.tolist())

# Convert to DataFrame
fixed_df = pd.DataFrame(fixed_rows)

# Optional: Replace bad characters
fixed_df = fixed_df.replace('â‚¹', '₹', regex=True)

# Save it
fixed_df.to_csv("cleaned_fixed_data.csv", index=False, header=False)
print("✅ Cleaned file saved as 'cleaned_fixed_data.csv'")

⚠️ Rows to split (probably merged listings): []
✅ Cleaned file saved as 'cleaned_fixed_data.csv'
